# AB01 Jinwoo RD — no-LPF replay test

Single-trial check: `ab01_jinwoo_knee_0p8mps_rd_exo_on`.

**No input LPF** anywhere in offline replay (contrast with deployed cascade / training pipeline):
- **Vicon IK oracle**: GPIO-synced processed IK angle + B-spline velocity → TCN (raw inputs)
- **Encoder replay**: logged `model_in_knee_angle_raw` + `model_in_knee_vel_raw` → TCN (raw inputs)
- **TCN output**: raw model output (no 6 Hz output LPF)

**GT** uses the standard scaled-ID path (OpenSim ID − cmd, 6 Hz zero-phase LPF, RD encoder–IK xcorr ID shift).

Checkpoint: `runs/0707_knee_finetune_balanced_lg_ra_rd/best_model.pt`

In [5]:
import io
import inspect
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy.signal import butter, sosfilt, sosfiltfilt

PROJECT_ROOT = Path('/home/metamobility3/Jinwoo/os_kinetics').resolve()
PROCESSED_SUBJECT_DIR = Path('/media/metamobility3/Samsung_T52/Results/AB01_Jinwoo_knee')
TELEMETRY_ROOT = PROJECT_ROOT

TRIAL_STEM = 'ab01_jinwoo_knee_0p8mps_rd_exo_on'
SUBJECT_TOKEN = 'ab01_jinwoo'
SUBJECT_NAME = 'AB01_Jinwoo'
SUBJECT_MASS_KG = 88.0

EXO_KIND = 'knee-exo'
JOINT = 'knee_angle_r'
MOMENT_COL = 'knee_angle_r_moment'
MOCAP_FS_HZ = 1000.0
GT_LPF_CUTOFF_HZ, GT_LPF_ORDER = 6.0, 4

CHECKPOINT_PATH = PROJECT_ROOT / 'runs' / '0707_knee_finetune_balanced_lg_ra_rd' / 'best_model.pt'
ID_ALIGN_XCORR_STEMS = frozenset({TRIAL_STEM})
ENCODER_IK_XCORR_MAX_LAG = 300

TRIM_START_SEC = 10.0
TRIM_END_SEC = 10.0

print(f'Trial: {TRIAL_STEM}')
print(f'Processed: {PROCESSED_SUBJECT_DIR}')
print(f'Checkpoint: {CHECKPOINT_PATH} ({"OK" if CHECKPOINT_PATH.is_file() else "MISSING"})')

Trial: ab01_jinwoo_knee_0p8mps_rd_exo_on
Processed: /media/metamobility3/Samsung_T52/Results/AB01_Jinwoo_knee
Checkpoint: /home/metamobility3/Jinwoo/os_kinetics/runs/0707_knee_finetune_balanced_lg_ra_rd/best_model.pt (OK)


In [6]:
def butter_lpf(x, fs_hz, cutoff_hz, order, mode='zero_phase'):
    arr = np.asarray(x, dtype=np.float64).reshape(-1)
    nyq = 0.5 * float(fs_hz)
    if cutoff_hz <= 0 or cutoff_hz >= nyq or len(arr) < 4:
        return arr
    sos = butter(int(order), float(cutoff_hz) / nyq, btype='low', output='sos')
    return sosfiltfilt(sos, arr) if mode == 'zero_phase' else sosfilt(sos, arr)


def lpf_nan(x, fs_hz, cutoff_hz, order, mode='zero_phase'):
    arr = np.asarray(x, dtype=np.float64).copy()
    finite = np.isfinite(arr)
    if finite.sum() < max(3, order + 1):
        return arr
    if finite.all():
        return butter_lpf(arr, fs_hz, cutoff_hz, order, mode)
    idx = np.arange(arr.size, dtype=np.float64)
    filled = np.interp(idx, idx[finite], arr[finite])
    out = butter_lpf(filled, fs_hz, cutoff_hz, order, mode)
    out[~finite] = np.nan
    return out


def rmse_r2(y_true, y_pred):
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    if m.sum() < 2:
        return np.nan, np.nan
    e = y_pred[m] - y_true[m]
    rmse = float(np.sqrt(np.mean(e ** 2)))
    ss_res = float(np.sum(e ** 2))
    ss_tot = float(np.sum((y_true[m] - np.mean(y_true[m])) ** 2))
    return rmse, float(1.0 - ss_res / (ss_tot + 1e-12))


def read_sto(path: Path):
    with open(path) as f:
        lines = f.readlines()
    end_idx = next(i for i, l in enumerate(lines) if l.strip().lower() == 'endheader')
    cols = lines[end_idx + 1].strip().split()
    data = np.loadtxt(io.StringIO(''.join(lines[end_idx + 2:])))
    if data.ndim == 1:
        data = data.reshape(1, -1)
    return cols, data


def infer_fs_hz(time_s, default_fs=100.0):
    if time_s is None or len(time_s) < 3:
        return float(default_fs)
    dt = np.diff(np.asarray(time_s, dtype=np.float64))
    dt = dt[np.isfinite(dt) & (dt > 0)]
    return float(1.0 / np.median(dt)) if dt.size else float(default_fs)


def trial_cond_speed(stem: str) -> Tuple[str, str]:
    parts = stem.lower().split('_')
    return parts[4].upper(), parts[3]


def resolve_trial_paths(trial_stem: str) -> Dict[str, Path]:
    cond, speed = trial_cond_speed(trial_stem)
    return {
        'npz': TELEMETRY_ROOT / f'{trial_stem}.npz',
        'mocap': PROCESSED_SUBJECT_DIR / EXO_KIND / 'mocap' / f'{cond}_{speed}.csv',
        'id': PROCESSED_SUBJECT_DIR / EXO_KIND / 'id' / f'{cond}_{speed}_id.sto',
        'ik': PROCESSED_SUBJECT_DIR / EXO_KIND / 'ik' / f'{cond}_{speed}_ik.mot',
        'cond': cond,
        'speed': speed,
    }


def parse_mocap_csv(path: Path, fs: float = MOCAP_FS_HZ):
    df = pd.read_csv(path, skiprows=[0, 1, 2, 4], header=0, low_memory=False, on_bad_lines='skip')
    df = df[pd.to_numeric(df['Frame'], errors='coerce').notna()].copy()
    df['jet'] = pd.to_numeric(df['jet'], errors='coerce')
    df = df.dropna(subset=['jet']).reset_index(drop=True)
    return np.arange(len(df)) / fs, df['jet'].to_numpy(dtype=float)


def normalize_gpio(gpio: np.ndarray) -> np.ndarray:
    arr = np.asarray(gpio, dtype=np.float64)
    g_range = arr.max() - arr.min()
    return arr if g_range <= 0 else (arr - arr.min()) / g_range


def first_falling_edge(signal: np.ndarray, threshold: float = 0.5) -> Optional[int]:
    above = np.asarray(signal, dtype=np.float64) > threshold
    for i in range(1, len(above)):
        if above[i - 1] and not above[i]:
            return i
    return None


def extract_gpio(npz) -> Tuple[np.ndarray, str]:
    for k in ('gpio_output', 'GPIO', 'gpio', 'trigger'):
        if k in npz.files:
            return np.asarray(npz[k], dtype=np.float64), k
    raise KeyError('No GPIO key in npz')


def extract_applied_cmd_nm(npz) -> Tuple[np.ndarray, str]:
    for k in ('cmd_R', 'cmd_L'):
        if k in npz.files:
            return np.asarray(npz[k], dtype=np.float64), k
    raise KeyError(f'No cmd_R/cmd_L; keys={sorted(npz.files)}')


def gpio_offset_s(t_exo, g_exo, t_mocap, g_mocap):
    idx_exo = first_falling_edge(g_exo)
    idx_mocap = first_falling_edge(normalize_gpio(g_mocap))
    if idx_exo is None or idx_mocap is None:
        return None, idx_exo, idx_mocap
    return float(t_mocap[idx_mocap] - t_exo[idx_exo]), idx_exo, idx_mocap


def _fill_nan_1d(x: np.ndarray) -> np.ndarray:
    arr = np.asarray(x, dtype=np.float64).copy()
    finite = np.isfinite(arr)
    if finite.all() or not finite.any():
        return arr
    arr[~finite] = np.interp(np.flatnonzero(~finite), np.flatnonzero(finite), arr[finite])
    return arr


def _shift_samples_1d(x: np.ndarray, lag_samples: int) -> np.ndarray:
    arr = np.asarray(x, dtype=np.float64)
    out = np.full_like(arr, np.nan)
    lag = int(lag_samples)
    if lag > 0:
        if lag < len(arr):
            out[:-lag] = arr[lag:]
    elif lag < 0:
        lag = -lag
        if lag < len(arr):
            out[lag:] = arr[:-lag]
    else:
        out = arr.copy()
    return _fill_nan_1d(out)


def sync_to_wave_t(t_src, y_src, wave: Dict, *, t_src_on_npz_clock: bool = False) -> np.ndarray:
    t_aligned = np.asarray(wave['t'], dtype=np.float64)
    y_src = np.asarray(y_src, dtype=np.float64)
    t_src = np.asarray(t_src, dtype=np.float64)
    if t_src_on_npz_clock:
        n = int(min(len(t_aligned), len(t_src), len(y_src)))
        t_aligned = t_aligned[:n]
        t_src = t_src[:n]
        y_src = y_src[:n]
        t_ref = t_src + float(wave['offset_s'])
    else:
        t_ref = t_src
    return _fill_nan_1d(np.interp(t_aligned, t_ref, y_src, left=np.nan, right=np.nan))


def analysis_trim_mask(t: np.ndarray, trim_start_s=TRIM_START_SEC, trim_end_s=TRIM_END_SEC) -> np.ndarray:
    t = np.asarray(t, dtype=np.float64)
    t_rel = t - np.nanmin(t)
    return (t_rel >= trim_start_s) & (t_rel <= (np.nanmax(t_rel) - trim_end_s))


def load_gpio_sync_data(trial_stem: str) -> Dict:
    paths = resolve_trial_paths(trial_stem)
    for key in ('npz', 'mocap', 'id', 'ik'):
        if not paths[key].exists():
            raise FileNotFoundError(paths[key])
    d = np.load(str(paths['npz']), allow_pickle=True)
    gpio, gpio_key = extract_gpio(d)
    t_raw = np.asarray(d['time'], dtype=np.float64) if 'time' in d.files else np.arange(len(gpio), dtype=np.float64)
    n = min(len(t_raw), len(gpio))
    t_raw, gpio = t_raw[:n], gpio[:n]
    t_mocap, gpio_mocap = parse_mocap_csv(paths['mocap'])
    offset_s, idx_exo, idx_mocap = gpio_offset_s(t_raw, gpio, t_mocap, gpio_mocap)
    return {
        'trial': trial_stem,
        'paths': paths,
        'npz': d,
        'gpio_key': gpio_key,
        'offset_s': offset_s,
        't_npz': t_raw,
        't_npz_aligned': t_raw + float(offset_s) if offset_s is not None else t_raw,
    }


def _load_vicon_knee_ik_deg(stem: str) -> Tuple[np.ndarray, np.ndarray]:
    paths = resolve_trial_paths(stem)
    cols, data = read_sto(paths['ik'])
    t_mocap = data[:, cols.index('time')].astype(np.float64)
    knee_deg = data[:, cols.index('knee_angle_r')].astype(np.float64)
    return t_mocap, knee_deg


def _vicon_ik_velocity_spline(t_mocap, knee_rad):
    from scipy.interpolate import splrep, splev
    if len(t_mocap) < 5:
        dt = 1.0 / infer_fs_hz(t_mocap)
        vel = np.zeros_like(knee_rad)
        if len(knee_rad) > 1:
            vel[1:] = (knee_rad[1:] - knee_rad[:-1]) / dt
        return vel
    tck = splrep(t_mocap, knee_rad, s=0, k=3)
    return np.asarray(splev(t_mocap, tck, der=1), dtype=np.float64)


def _load_synced_encoder_angle_deg(trial_stem: str, sync: Dict, n: int) -> Tuple[np.ndarray, str]:
    d = sync['npz']
    t_npz = np.asarray(sync['t_npz'][:n], dtype=np.float64)
    wave_stub = {'t': sync['t_npz_aligned'][:n], 'offset_s': sync['offset_s']}
    for key in ('model_in_knee_angle_raw', 'knee_angle_r'):
        if key in d.files:
            enc_raw = np.asarray(d[key][:n], dtype=np.float64)
            enc_sync = sync_to_wave_t(t_npz, enc_raw, wave_stub, t_src_on_npz_clock=True)
            return np.rad2deg(enc_sync), key
    raise KeyError('No encoder angle')


def _load_synced_vicon_ik_angle_deg(trial_stem: str, sync: Dict, n: int) -> np.ndarray:
    t_mocap, knee_deg = _load_vicon_knee_ik_deg(trial_stem)
    wave_stub = {'t': sync['t_npz_aligned'][:n], 'offset_s': sync['offset_s']}
    return sync_to_wave_t(t_mocap, knee_deg, wave_stub, t_src_on_npz_clock=False)


def compute_encoder_ik_xcorr_lag_samples(trial_stem: str, sync: Dict, n: int) -> Tuple[int, str]:
    enc_deg, enc_key = _load_synced_encoder_angle_deg(trial_stem, sync, n)
    vicon_deg = _load_synced_vicon_ik_angle_deg(trial_stem, sync, n)
    trim_m = analysis_trim_mask(sync['t_npz_aligned'][:n])
    best_lag, best_score = 0, -np.inf
    for lag in range(-ENCODER_IK_XCORR_MAX_LAG, ENCODER_IK_XCORR_MAX_LAG + 1):
        shifted = _shift_samples_1d(enc_deg, lag)
        s, v = shifted[trim_m], vicon_deg[trim_m]
        mm = np.isfinite(s) & np.isfinite(v)
        if mm.sum() < 100:
            continue
        score = float(np.corrcoef(s[mm], v[mm])[0, 1])
        if score > best_score:
            best_score, best_lag = score, lag
    return int(best_lag), enc_key


def load_gt_waveform(trial_stem: str, sync: Dict) -> Dict:
    paths = sync['paths']
    d = sync['npz']
    applied_nm, applied_key = extract_applied_cmd_nm(d)
    n = min(len(sync['t_npz']), len(applied_nm))
    t_aligned = sync['t_npz_aligned'][:n].astype(np.float64)
    applied_nm = applied_nm[:n]
    fs_hz = infer_fs_hz(sync['t_npz'][:n])
    id_shift = 0
    encoder_key = ''
    if trial_stem in ID_ALIGN_XCORR_STEMS:
        id_shift, encoder_key = compute_encoder_ik_xcorr_lag_samples(trial_stem, sync, n)
    cols, id_data = read_sto(paths['id'])
    t_id = id_data[:, cols.index('time')]
    id_moment_nm = id_data[:, cols.index(MOMENT_COL)]
    t_id_query = t_aligned + id_shift / fs_hz
    id_nm_raw = np.interp(t_id_query, t_id, id_moment_nm, left=np.nan, right=np.nan)
    net_raw_nmpkg = id_nm_raw / SUBJECT_MASS_KG - applied_nm / SUBJECT_MASS_KG
    gt_nmpkg = lpf_nan(net_raw_nmpkg, fs_hz, GT_LPF_CUTOFF_HZ, GT_LPF_ORDER, 'zero_phase')
    return {
        't': t_aligned,
        'gt_nmpkg': gt_nmpkg,
        'fs_hz': fs_hz,
        'offset_s': float(sync['offset_s']),
        'applied_key': applied_key,
        'encoder_ik_xcorr_lag_samples': id_shift,
        'encoder_angle_key': encoder_key,
    }


def load_synced_encoder_imu_raw(stem: str, wave: Dict) -> Tuple[np.ndarray, np.ndarray, str, str]:
    d = np.load(str(TELEMETRY_ROOT / f'{stem}.npz'), allow_pickle=True)
    t_npz = np.asarray(d['time'], dtype=np.float64)
    angle_key = vel_key = None
    angle_raw = vel_raw = None
    for key in ('model_in_knee_angle_raw', 'knee_angle_r'):
        if key in d.files:
            angle_raw = np.asarray(d[key], dtype=np.float64)
            angle_key = key
            break
    for key in ('model_in_knee_vel_raw', 'model_in_knee_vel_raw_r'):
        if key in d.files:
            vel_raw = np.asarray(d[key], dtype=np.float64)
            vel_key = key
            break
    if vel_raw is None and 'gyro_shank_r' in d.files and 'gyro_thigh_r' in d.files:
        vel_raw = np.asarray(d['gyro_shank_r'], dtype=np.float64) - np.asarray(d['gyro_thigh_r'], dtype=np.float64)
        vel_key = 'gyro_shank_r-gyro_thigh_r'
    if angle_raw is None or vel_raw is None:
        raise KeyError(f'Missing encoder/IMU in {stem}')
    return (
        sync_to_wave_t(t_npz, angle_raw, wave, t_src_on_npz_clock=True),
        sync_to_wave_t(t_npz, vel_raw, wave, t_src_on_npz_clock=True),
        angle_key,
        vel_key,
    )


def load_synced_vicon_ik_raw(stem: str, wave: Dict) -> Tuple[np.ndarray, np.ndarray]:
    t_mocap, knee_deg = _load_vicon_knee_ik_deg(stem)
    knee_rad = np.deg2rad(knee_deg)
    knee_vel = _vicon_ik_velocity_spline(t_mocap, knee_rad)
    angle_sync = sync_to_wave_t(t_mocap, knee_rad, wave, t_src_on_npz_clock=False)
    vel_sync = sync_to_wave_t(t_mocap, knee_vel, wave, t_src_on_npz_clock=False)
    return angle_sync, vel_sync


sys.path.insert(0, str(PROJECT_ROOT))
from model import TCN  # noqa: E402


def _tcn_ctor_kwargs(cfg: dict) -> dict:
    allowed = {k for k in inspect.signature(TCN.__init__).parameters if k != 'self'}
    return {k: v for k, v in cfg.items() if k in allowed}


@torch.no_grad()
def run_knee_tcn_inference(model, angle, vel, window_size, device):
    angle = np.asarray(angle, dtype=np.float32)
    vel = np.asarray(vel, dtype=np.float32)
    n = int(min(len(angle), len(vel)))
    pred = np.zeros(n, dtype=np.float32)
    model.eval()
    for t in range(n):
        start = max(0, t - window_size + 1)
        valid = t - start + 1
        x = np.zeros((2, window_size), dtype=np.float32)
        x[0, -valid:] = angle[start : t + 1]
        x[1, -valid:] = vel[start : t + 1]
        xt = torch.from_numpy(x).unsqueeze(0).to(device=device, dtype=torch.float32)
        pred[t] = float(model(xt)[0, 0, -1].item())
    return pred


def load_replay_model(ckpt_path: Path, device: str):
    ckpt = torch.load(str(ckpt_path), map_location='cpu', weights_only=False)
    model_cfg = ckpt['model_config']
    window_size = int(ckpt.get('window_size', 100))
    model = TCN(**_tcn_ctor_kwargs(model_cfg)).eval()
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(device)
    return model, window_size

print('Helpers ready.')

Helpers ready.


In [7]:
sync = load_gpio_sync_data(TRIAL_STEM)
if sync['offset_s'] is None:
    raise RuntimeError('GPIO sync failed')

base = load_gt_waveform(TRIAL_STEM, sync)
wave = dict(base)
wave['trial'] = TRIAL_STEM
wave['offset_s'] = float(sync['offset_s'])

vicon_angle_rad, vicon_vel_rad_s = load_synced_vicon_ik_raw(TRIAL_STEM, wave)
enc_angle_rad, enc_vel_rad_s, enc_angle_key, enc_vel_key = load_synced_encoder_imu_raw(TRIAL_STEM, wave)

n = int(min(len(wave['t']), len(vicon_angle_rad), len(enc_angle_rad), len(wave['gt_nmpkg'])))
for key in ('t', 'gt_nmpkg'):
    wave[key] = np.asarray(wave[key], dtype=np.float64)[:n]
vicon_angle_rad = vicon_angle_rad[:n]
vicon_vel_rad_s = vicon_vel_rad_s[:n]
enc_angle_rad = enc_angle_rad[:n]
enc_vel_rad_s = enc_vel_rad_s[:n]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, window_size = load_replay_model(CHECKPOINT_PATH, device)

vicon_pred = run_knee_tcn_inference(model, vicon_angle_rad, vicon_vel_rad_s, window_size, device).astype(np.float64)
enc_pred = run_knee_tcn_inference(model, enc_angle_rad, enc_vel_rad_s, window_size, device).astype(np.float64)

TEST_DATA = {
    **wave,
    'vicon_angle_rad': vicon_angle_rad,
    'vicon_vel_rad_s': vicon_vel_rad_s,
    'encoder_angle_rad': enc_angle_rad,
    'encoder_vel_rad_s': enc_vel_rad_s,
    'encoder_angle_key': enc_angle_key,
    'encoder_vel_key': enc_vel_key,
    'vicon_replay_nmpkg': vicon_pred,
    'encoder_replay_nmpkg': enc_pred,
}

trim_m = analysis_trim_mask(TEST_DATA['t'])
gt = TEST_DATA['gt_nmpkg'][trim_m]
metrics = []
for label, col_angle, col_vel, col_out in (
    ('inputs', 'vicon_angle_rad', 'vicon_vel_rad_s', None),
    ('Vicon IK replay (no LPF)', 'vicon_angle_rad', 'vicon_vel_rad_s', 'vicon_replay_nmpkg'),
    ('Encoder replay (no LPF)', 'encoder_angle_rad', 'encoder_vel_rad_s', 'encoder_replay_nmpkg'),
):
    row = {'series': label}
    if col_out is None:
        row['rmse_angle_deg'] = rmse_r2(
            np.rad2deg(TEST_DATA['vicon_angle_rad'][trim_m]),
            np.rad2deg(TEST_DATA['encoder_angle_rad'][trim_m]),
        )[0]
        row['rmse_vel_rad_s'] = rmse_r2(
            TEST_DATA['vicon_vel_rad_s'][trim_m], TEST_DATA['encoder_vel_rad_s'][trim_m]
        )[0]
    else:
        pred = TEST_DATA[col_out][trim_m]
        rmse, r2 = rmse_r2(gt, pred)
        row['rmse_nmpkg'] = rmse
        row['r2'] = r2
    metrics.append(row)

metrics_df = pd.DataFrame(metrics)
print(
    f"GPIO offset={TEST_DATA['offset_s']:+.3f}s | "
    f"ID xcorr shift={TEST_DATA['encoder_ik_xcorr_lag_samples']:+d} samples | "
    f"fs={TEST_DATA['fs_hz']:.1f} Hz | window={window_size} | device={device}"
)
print(f"Encoder keys: angle={enc_angle_key} | vel={enc_vel_key}")
display(metrics_df)

GPIO offset=-0.120s | ID xcorr shift=+2 samples | fs=100.0 Hz | window=100 | device=cuda
Encoder keys: angle=model_in_knee_angle_raw | vel=model_in_knee_vel_raw


,series,rmse_angle_deg,rmse_vel_rad_s,rmse_nmpkg,r2
0,inputs,4.052182,1.025659,NaN,NaN
1,Vicon IK replay (no LPF),NaN,NaN,0.215300,0.688907
2,Encoder replay (no LPF),NaN,NaN,0.273099,0.499456


In [8]:
COLORS = {
    'GT': '#424242',
    'Vicon IK': '#2e7d32',
    'Encoder': '#1565c0',
}

plot_out = widgets.Output()
time_slider = widgets.FloatRangeSlider(
    description='Time (s):', continuous_update=False, readout_format='.2f', layout=widgets.Layout(width='760px')
)


def draw_no_lpf_test(wave: Dict, t_window: Tuple[float, float]) -> None:
    t_rel = wave['t'] - np.nanmin(wave['t'])
    m = analysis_trim_mask(wave['t']) & (t_rel >= t_window[0]) & (t_rel <= t_window[1])
    gt = wave['gt_nmpkg'][m]
    vicon_out = wave['vicon_replay_nmpkg'][m]
    enc_out = wave['encoder_replay_nmpkg'][m]
    rmse_v, r2_v = rmse_r2(gt, vicon_out)
    rmse_e, r2_e = rmse_r2(gt, enc_out)
    vicon_ang = np.rad2deg(wave['vicon_angle_rad'][m])
    enc_ang = np.rad2deg(wave['encoder_angle_rad'][m])
    rmse_ang, r2_ang = rmse_r2(vicon_ang, enc_ang)
    vicon_vel = wave['vicon_vel_rad_s'][m]
    enc_vel = wave['encoder_vel_rad_s'][m]
    rmse_vel, r2_vel = rmse_r2(vicon_vel, enc_vel)

    fig, axs = plt.subplots(3, 1, figsize=(14, 11), sharex=True)

    ax = axs[0]
    ax.plot(t_rel[m], gt, color=COLORS['GT'], lw=2.0, label='GT (ID/mass − cmd/mass, 6 Hz z-p LPF)')
    ax.plot(
        t_rel[m], vicon_out, color=COLORS['Vicon IK'], lw=1.5, ls='-',
        label=f'Vicon IK replay, no input/output LPF (RMSE={rmse_v:.3f}, R²={r2_v:.3f})',
    )
    ax.plot(
        t_rel[m], enc_out, color=COLORS['Encoder'], lw=1.5, ls='-.',
        label=f'Encoder+IMU replay, no input/output LPF (RMSE={rmse_e:.3f}, R²={r2_e:.3f})',
    )
    ax.set_ylabel('N·m/kg')
    ax.set_title('Model output vs GT (raw TCN, no input LPF)')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.25)

    ax = axs[1]
    ax.plot(t_rel[m], vicon_ang, color=COLORS['Vicon IK'], lw=1.5, label='Vicon IK knee_angle_r (synced, no LPF)')
    ax.plot(
        t_rel[m], enc_ang, color=COLORS['Encoder'], lw=1.5, ls='-.',
        label=f"Encoder ({wave['encoder_angle_key']}, logged raw)",
    )
    ax.set_ylabel('Knee angle (deg)')
    ax.set_title(f'Angle inputs | RMSE={rmse_ang:.2f}° R²={r2_ang:.3f}')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.25)

    ax = axs[2]
    ax.plot(t_rel[m], vicon_vel, color=COLORS['Vicon IK'], lw=1.5, label='Vicon IK dθ/dt (spline, no LPF)')
    ax.plot(
        t_rel[m], enc_vel, color=COLORS['Encoder'], lw=1.5, ls='-.',
        label=f"IMU ({wave['encoder_vel_key']}, logged raw)",
    )
    ax.set_xlabel('Time since sync (s)')
    ax.set_ylabel('Knee angular velocity (rad/s)')
    ax.set_title(f'Velocity inputs | RMSE={rmse_vel:.3f} rad/s R²={r2_vel:.3f}')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.25)

    fig.suptitle(
        f'AB01 Jinwoo RD | GPIO={wave["offset_s"]:+.3f}s | ID shift={wave["encoder_ik_xcorr_lag_samples"]:+d} samples',
        y=1.01,
    )
    fig.tight_layout()
    with plot_out:
        plot_out.clear_output(wait=True)
        plt.show()


def _init_slider():
    t_rel = TEST_DATA['t'] - np.nanmin(TEST_DATA['t'])
    m = analysis_trim_mask(TEST_DATA['t'])
    t_use = t_rel[m] if m.any() else t_rel
    time_slider.min = float(t_use[0])
    time_slider.max = float(t_use[-1])
    time_slider.step = max((time_slider.max - time_slider.min) / 500, 1e-3)
    time_slider.value = (time_slider.min, time_slider.max)


def _redraw(*_):
    draw_no_lpf_test(TEST_DATA, time_slider.value)


time_slider.observe(_redraw, names='value')
_init_slider()
display(widgets.VBox([time_slider, plot_out]))
_redraw()

## Zero-phase LPF comparison

Apply the **same 6 Hz zero-phase Butterworth LPF (order 4)** that GT already uses to the two raw TCN outputs, so all three curves are filtered identically:

- **GT** — 6 Hz zero-phase LPF (unchanged; already filtered)
- **Vicon IK replay** — raw TCN output → 6 Hz zero-phase LPF
- **Encoder replay** — raw TCN output → 6 Hz zero-phase LPF

Compares RMSE/R² of each replay vs GT, no-LPF vs LPF.

In [9]:
fs_hz = TEST_DATA['fs_hz']
TEST_DATA['vicon_replay_lpf_nmpkg'] = lpf_nan(
    TEST_DATA['vicon_replay_nmpkg'], fs_hz, GT_LPF_CUTOFF_HZ, GT_LPF_ORDER, 'zero_phase'
)
TEST_DATA['encoder_replay_lpf_nmpkg'] = lpf_nan(
    TEST_DATA['encoder_replay_nmpkg'], fs_hz, GT_LPF_CUTOFF_HZ, GT_LPF_ORDER, 'zero_phase'
)

trim_m = analysis_trim_mask(TEST_DATA['t'])
gt = TEST_DATA['gt_nmpkg'][trim_m]
lpf_metrics = []
for label, raw_col, lpf_col in (
    ('Vicon IK replay', 'vicon_replay_nmpkg', 'vicon_replay_lpf_nmpkg'),
    ('Encoder replay', 'encoder_replay_nmpkg', 'encoder_replay_lpf_nmpkg'),
):
    rmse_raw, r2_raw = rmse_r2(gt, TEST_DATA[raw_col][trim_m])
    rmse_lpf, r2_lpf = rmse_r2(gt, TEST_DATA[lpf_col][trim_m])
    lpf_metrics.append({
        'series': label,
        'rmse_no_lpf': rmse_raw,
        'r2_no_lpf': r2_raw,
        'rmse_lpf': rmse_lpf,
        'r2_lpf': r2_lpf,
        'd_rmse': rmse_lpf - rmse_raw,
        'd_r2': r2_lpf - r2_raw,
    })
lpf_metrics_df = pd.DataFrame(lpf_metrics)
print(
    f'Zero-phase Butterworth LPF: cutoff={GT_LPF_CUTOFF_HZ} Hz, order={GT_LPF_ORDER}, fs={fs_hz:.1f} Hz'
)
print('GT already uses this exact LPF; now applied to both raw TCN outputs so all three match.')
display(lpf_metrics_df)

Zero-phase Butterworth LPF: cutoff=6.0 Hz, order=4, fs=100.0 Hz
GT already uses this exact LPF; now applied to both raw TCN outputs so all three match.


,series,rmse_no_lpf,r2_no_lpf,rmse_lpf,r2_lpf,d_rmse,d_r2
0,Vicon IK replay,0.215300,0.688907,0.185956,0.767928,-0.029344,0.079021
1,Encoder replay,0.273099,0.499456,0.216325,0.685939,-0.056775,0.186484


In [ ]:
lpf_plot_out = widgets.Output()
lpf_time_slider = widgets.FloatRangeSlider(
    description='Time (s):', continuous_update=False, readout_format='.2f',
    layout=widgets.Layout(width='760px'),
)


def draw_lpf_test(wave: Dict, t_window: Tuple[float, float]) -> None:
    t_rel = wave['t'] - np.nanmin(wave['t'])
    m = analysis_trim_mask(wave['t']) & (t_rel >= t_window[0]) & (t_rel <= t_window[1])
    gt = wave['gt_nmpkg'][m]
    vicon_raw = wave['vicon_replay_nmpkg'][m]
    enc_raw = wave['encoder_replay_nmpkg'][m]
    vicon_lpf = wave['vicon_replay_lpf_nmpkg'][m]
    enc_lpf = wave['encoder_replay_lpf_nmpkg'][m]
    rmse_v, r2_v = rmse_r2(gt, vicon_lpf)
    rmse_e, r2_e = rmse_r2(gt, enc_lpf)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(t_rel[m], gt, color=COLORS['GT'], lw=2.0, label='GT (6 Hz z-p LPF)')
    ax.plot(t_rel[m], vicon_raw, color=COLORS['Vicon IK'], lw=0.8, alpha=0.3, label='Vicon replay (raw)')
    ax.plot(t_rel[m], enc_raw, color=COLORS['Encoder'], lw=0.8, alpha=0.3, label='Encoder replay (raw)')
    ax.plot(
        t_rel[m], vicon_lpf, color=COLORS['Vicon IK'], lw=1.6,
        label=f'Vicon replay + 6 Hz z-p LPF (RMSE={rmse_v:.3f}, R²={r2_v:.3f})',
    )
    ax.plot(
        t_rel[m], enc_lpf, color=COLORS['Encoder'], lw=1.6, ls='-.',
        label=f'Encoder replay + 6 Hz z-p LPF (RMSE={rmse_e:.3f}, R²={r2_e:.3f})',
    )
    ax.set_xlabel('Time since sync (s)')
    ax.set_ylabel('N·m/kg')
    ax.set_title('Model output + 6 Hz zero-phase LPF vs GT (all three filtered identically)')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.25)
    fig.suptitle(
        f'AB01 Jinwoo RD | GPIO={wave["offset_s"]:+.3f}s | ID shift={wave["encoder_ik_xcorr_lag_samples"]:+d} samples',
        y=1.02,
    )
    fig.tight_layout()
    with lpf_plot_out:
        lpf_plot_out.clear_output(wait=True)
        plt.show()


def _init_lpf_slider():
    t_rel = TEST_DATA['t'] - np.nanmin(TEST_DATA['t'])
    m = analysis_trim_mask(TEST_DATA['t'])
    t_use = t_rel[m] if m.any() else t_rel
    lpf_time_slider.min = float(t_use[0])
    lpf_time_slider.max = float(t_use[-1])
    lpf_time_slider.step = max((lpf_time_slider.max - lpf_time_slider.min) / 500, 1e-3)
    lpf_time_slider.value = (lpf_time_slider.min, lpf_time_slider.max)


def _redraw_lpf(*_):
    draw_lpf_test(TEST_DATA, lpf_time_slider.value)


lpf_time_slider.observe(_redraw_lpf, names='value')
_init_lpf_slider()
display(widgets.VBox([lpf_time_slider, lpf_plot_out]))
_redraw_lpf()